# Step 2: does the on-device ONNX model give the same answers?

The app plans to bundle `Xenova/clap-htsat-unfused` (ONNX) and run it on the phone. Step 1 used the original PyTorch model.
This notebook checks, on the same 440 CatMeows clips and the same input features:

1. What the ONNX audio files expect and return (names, shapes, sizes).
2. Whether ONNX embeddings match PyTorch (cosine similarity per clip), for the full-precision and the quantized file.
3. Whether the step 1 classifier result (macro F1 0.49, 7-fold by cat) still holds with ONNX embeddings.
4. How long one clip takes with onnxruntime on CPU.

Out of scope here: computing the log-mel features in Dart (that is the next step).

In [ ]:
!pip -q install onnxruntime
import os, glob, json, zipfile, urllib.request, time
import numpy as np, librosa, torch, onnxruntime as ort, transformers
from huggingface_hub import hf_hub_download, list_repo_files
from transformers import ClapModel, ClapProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score
SR = 48000
print("transformers", transformers.__version__, "| onnxruntime", ort.__version__)

## 1. Same data as step 1

In [ ]:
if not os.path.isdir("catmeows"):
    rec = json.load(urllib.request.urlopen("https://zenodo.org/api/records/4008297"))
    zf = [f for f in rec["files"] if f["key"] == "dataset.zip"][0]
    urllib.request.urlretrieve(zf["links"]["self"], "catmeows.zip")
    zipfile.ZipFile("catmeows.zip").extractall("catmeows")
files = sorted(glob.glob("catmeows/**/*.wav", recursive=True))
classes = ["B", "F", "I"]
y = np.array([classes.index(os.path.basename(f).split("_")[0]) for f in files])
cats = np.array([os.path.basename(f).split("_")[1] for f in files])
audio = [librosa.load(f, sr=SR, mono=True)[0] for f in files]
print(len(files), "clips,", len(set(cats)), "cats")

## 2. Input features (computed once, fed to both models)

In [ ]:
proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
fe = proc.feature_extractor
feats = np.concatenate([fe(audio[i:i+16], sampling_rate=SR, padding="repeatpad", return_tensors="np")["input_features"]
                        for i in range(0, len(audio), 16)]).astype(np.float32)
print("input_features:", feats.shape, feats.dtype)
print("extractor settings:", {k: getattr(fe, k, None) for k in ["sampling_rate","fft_window_size","hop_length","feature_size","frequency_min","frequency_max","max_length_s","padding","truncation"]})

# does Xenova ship the same preprocessing settings?
xcfg = json.load(open(hf_hub_download("Xenova/clap-htsat-unfused", "preprocessor_config.json")))
lcfg = json.load(open(hf_hub_download("laion/clap-htsat-unfused", "preprocessor_config.json")))
diff = {k: (lcfg.get(k), xcfg.get(k)) for k in sorted(set(lcfg) | set(xcfg)) if lcfg.get(k) != xcfg.get(k)}
print("preprocessor differences laion vs Xenova:", diff or "none")

## 3. PyTorch embeddings (reference, as in step 1)

In [ ]:
model = ClapModel.from_pretrained("laion/clap-htsat-unfused").eval()
t0 = time.time()
with torch.no_grad():
    ref = np.concatenate([model.audio_projection(model.audio_model(input_features=torch.from_numpy(feats[i:i+16])).pooler_output).numpy()
                          for i in range(0, len(feats), 16)])
print("PyTorch:", ref.shape, f"{time.time()-t0:.0f} s")

## 4. The ONNX files

In [ ]:
onnx_files = [f for f in list_repo_files("Xenova/clap-htsat-unfused") if f.startswith("onnx/audio_model")]
print("audio ONNX files on the Hub:", onnx_files)
wanted = [f for f in ["onnx/audio_model.onnx", "onnx/audio_model_quantized.onnx"] if f in onnx_files]
sessions = {}
for f in wanted:
    p = hf_hub_download("Xenova/clap-htsat-unfused", f)
    s = ort.InferenceSession(p, providers=["CPUExecutionProvider"])
    sessions[f] = s
    print(f"\n{f}: {os.path.getsize(p)/1e6:.1f} MB")
    for i in s.get_inputs():  print("  input ", i.name, i.shape, i.type)
    for o in s.get_outputs(): print("  output", o.name, o.shape, o.type)

In [ ]:
def run_onnx(s, x):
    feed = {}
    for i in s.get_inputs():
        if i.name == "input_features": feed[i.name] = x
        elif "bool" in i.type:         feed[i.name] = np.zeros((x.shape[0], 1), dtype=bool)
        else: raise ValueError("unexpected input " + i.name)
    outs = dict(zip([o.name for o in s.get_outputs()], s.run(None, feed)))
    # the projected 512-d embedding is the output whose last dimension matches PyTorch
    name = "audio_embeds" if "audio_embeds" in outs else [k for k, v in outs.items() if v.ndim == 2 and v.shape[-1] == ref.shape[-1]][0]
    return outs[name], name

def l2(a): return a / np.linalg.norm(a, axis=1, keepdims=True)
def lr_cv(X):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=5000)), l2(X), y, groups=cats, cv=GroupKFold(n_splits=7))
    return round(f1_score(y, p, average="macro"), 3), round(accuracy_score(y, p), 3)

results = {"pytorch": {"macro_f1_cv7": lr_cv(ref)[0], "accuracy_cv7": lr_cv(ref)[1]}}
for f, s in sessions.items():
    t0 = time.time()
    emb = np.concatenate([run_onnx(s, feats[i:i+16])[0] for i in range(0, len(feats), 16)])
    name = run_onnx(s, feats[:1])[1]
    cos = (l2(emb) * l2(ref)).sum(1)
    t1 = time.time()
    for _ in range(5): run_onnx(s, feats[:1])
    per_clip = (time.time() - t1) / 5
    f1, acc = lr_cv(emb)
    results[f] = {"output_used": name, "cosine_vs_pytorch_min": round(float(cos.min()), 4), "cosine_vs_pytorch_mean": round(float(cos.mean()), 4),
                  "macro_f1_cv7": f1, "accuracy_cv7": acc, "seconds_per_clip_cpu": round(per_clip, 3)}
for k, v in results.items(): print(k, v)
print("\nRESULTS_JSON", json.dumps(results))